In [1]:
# ===========================
# Imports
# ===========================
import os
import pickle
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
from lightning_fabric.utilities.seed import seed_everything
from lightning.pytorch import Trainer
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss, MAE, RMSE
from sklearn.metrics import r2_score


# ===========================
# TFTPipeline
# ===========================
class TFTPipeline:
    def __init__(self, csv_path, test_year=2022, series_id_value="France"):
        torch.set_float32_matmul_precision('medium')
        self.csv_path = csv_path
        self.test_year = test_year
        self.series_id_value = series_id_value

        self.df = None
        #self.df_train = None
        #self.df_test = None

        self.training = None
        self.validation = None
        self.train_dataloader = None
        self.val_dataloader = None

        self.model = None
        self.trainer = None
        self.logger = None
        self.writer = None

    def load_and_prepare_data(self, params):
        df = pd.read_csv(self.csv_path)
        df["Datetime"] = pd.to_datetime(df["Datetime"])
        df = df.sort_values("Datetime").reset_index(drop=True)

        df["time_idx"] = range(len(df))
        df["series_id"] = self.series_id_value

        self.df = df

        df = df.set_index("Datetime")
        #self.df_train = df[df.index.year < self.test_year].reset_index()
        #self.df_test = df[df.index.year == self.test_year].reset_index()

    def setup_datasets(self, params):
        max_encoder_length = params.get("max_encoder_length", 48)
        max_prediction_length = params.get("max_prediction_length", 48)
        batch_size = params.get("batch_size", 64)

        training_cutoff = self.df["time_idx"].max() - max_prediction_length * 365

        known_reals = params.get("known_reals", ["time_idx"])
        unknown_reals = params.get("unknown_reals", ["Consommation"])
        known_categoricals = params.get("known_categoricals", [])
        unknown_categoricals = params.get("unknown_categoricals", [])
        static_reals = params.get("static_reals", [])
        static_categoricals = params.get("static_categoricals", [])

        self.training = TimeSeriesDataSet(
            self.df[self.df.time_idx <= training_cutoff],
            time_idx="time_idx",
            target="Consommation",
            group_ids=["series_id"],
            max_encoder_length=max_encoder_length,
            max_prediction_length=max_prediction_length,
            time_varying_known_reals=known_reals,
            time_varying_unknown_reals=unknown_reals,
            time_varying_known_categoricals=known_categoricals,
            time_varying_unknown_categoricals=unknown_categoricals,
            static_categoricals=static_categoricals,
            static_reals=static_reals,
            target_normalizer=params.get("target_normalizer", None),
            add_relative_time_idx=params.get("add_relative_time_idx", True),
            add_target_scales=params.get("add_target_scales", True),
            add_encoder_length=params.get("add_encoder_length", True),
        )

        self.validation = TimeSeriesDataSet.from_dataset(
            self.training, self.df, predict=True, stop_randomization=True
        )

        self.train_dataloader = self.training.to_dataloader(
            train=True,
            batch_size=batch_size,
            num_workers=params.get("train_num_workers", 16)
        )
        self.val_dataloader = self.validation.to_dataloader(
            train=False,
            batch_size=batch_size * 10,
            num_workers=params.get("val_num_workers", 16)
        )

    def save_datasets(self):
        path = os.path.join(self.logger.log_dir, "ts_datasets")
        os.makedirs(path, exist_ok=True)
        with open(os.path.join(path, "training.pkl"), "wb") as f:
            pickle.dump(self.training, f)
        with open(os.path.join(path, "validation.pkl"), "wb") as f:
            pickle.dump(self.validation, f)

    def build_model(self, params):
        self.model = TemporalFusionTransformer.from_dataset(
            self.training,
            learning_rate=params.get("learning_rate", 0.001),
            hidden_size=params.get("hidden_size", 128),
            attention_head_size=params.get("attention_head_size", 4),
            dropout=params.get("dropout", 0.1),
            hidden_continuous_size=params.get("hidden_continuous_size", 64),
            output_size=params.get("output_size", 3),
            loss=QuantileLoss(quantiles=[0.025, 0.50, 0.975])
        )

    def train_model(self, params):
        self.logger = TensorBoardLogger(
            save_dir=params.get("logger_dir", "lightning_logs"),
            name=params.get("logger_name", "tft_model")
        )

        callbacks = []
        if params.get("use_early_stopping", False):
            callbacks.append(EarlyStopping(
                monitor="val_loss",
                patience=params.get("early_stopping_patience", 5),
                mode="min"
            ))
        if params.get("use_checkpoint", False):
            callbacks.append(ModelCheckpoint(
                save_top_k=-1,
                monitor="val_loss",
                mode="min",
                dirpath=os.path.join(self.logger.log_dir, "Checkpoint"),
                filename="{epoch:02d}-{val_loss:.2f}"
            ))

        self.trainer = Trainer(
            max_epochs=params.get("max_epochs", 20),
            log_every_n_steps=params.get("log_every_n_steps", 1),
            accelerator=params.get("accelerator", "gpu"),
            devices=params.get("devices", 1),
            precision=params.get("precision", "32-true"),
            gradient_clip_val=params.get("gradient_clip_val", 0.1),
            logger=self.logger,
            callbacks=callbacks,
        )

        self.trainer.fit(
            self.model,
            train_dataloaders=self.train_dataloader,
            val_dataloaders=self.val_dataloader
        )

        self.save_datasets()

    def evaluate_model(self, params):
        log_dir = params.get("logger_dir", "lightning_logs") + "/" + params.get("logger_name", "tft_model")
        self.writer = SummaryWriter(log_dir=log_dir)

        raw_predictions = self.model.predict(
            self.val_dataloader,
            mode="raw",
            return_x=True
        )

        pred_2_5 = raw_predictions.output.prediction[:, :, 0]
        pred_97_5 = raw_predictions.output.prediction[:, :, 2]
        pred_median = raw_predictions.output.prediction[:, :, 1]
        true_values = raw_predictions.x["decoder_target"]

        mae = MAE()(pred_97_5, true_values)
        rmse = RMSE()(pred_97_5, true_values)
        r2 = r2_score(true_values.detach().cpu().numpy().flatten(), pred_97_5.detach().cpu().numpy().flatten())

        is_covered = ((true_values >= pred_2_5) & (true_values <= pred_97_5)).float()
        coverage = is_covered.mean()

        print(f"MAE: {mae.item():.2f}")
        print(f"RMSE: {rmse.item():.2f}")
        print(f"R2: {r2:.2f}")
        print(f"Coverage (entre 2.5%-97.5%): {coverage.item() * 100:.2f}%")

        self.writer.add_scalar("Validation/MAE", mae.item(), 0)
        self.writer.add_scalar("Validation/RMSE", rmse.item(), 0)
        self.writer.add_scalar("Validation/R2", r2, 0)
        self.writer.add_scalar("Validation/Coverage", coverage.item() * 100, 0)

        self.writer.close()

    def run_pipeline(self, params):
        seed = params.get("seed", 42)
        seed_everything(seed, workers=True)
        self.load_and_prepare_data(params)
        self.setup_datasets(params)
        self.build_model(params)
        self.train_model(params)
        self.evaluate_model(params)

In [2]:
params = {
    "max_encoder_length": 6*48,
    "max_prediction_length": 48,
    "max_epochs": 50,
    "dropout": 0.30,
    "seed": 84,
    "batch_size": 128,
    "learning_rate": 0.001,
    "hidden_size": 256,
    "attention_head_size": 8,
    "gradient_clip_val": 0.1,
    "hidden_continuous_size": 64,
    "accelerator": "gpu",
    "devices": 1,
    "known_reals": ["time_idx", "temperature", "is_vacation"],
    "unknown_reals": ["Consommation", "sunshine_duration"],
    "known_categoricals": ["weekday", "month", "season"],
    "log_every_n_steps": 1,
    "use_early_stopping": True,
    "early_stopping_patience": 10,
    "use_checkpoint": True
}

pipeline = TFTPipeline(csv_path="../data/03_primary/eco2mix/aggregate_minute/tempo_consumption_enriched_data.csv")
pipeline.run_pipeline(params=params)

Seed set to 84
/home/hassani/EPSI/MSPR/tft-env/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/home/hassani/EPSI/MSPR/tft-env/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss  

Sanity Checking: |                                                                                            …

Training: |                                                                                                   …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

... (21 outputs left out)

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


MAE: 4418.79
RMSE: 4626.13
R2: -1.81
Coverage (entre 2.5%-97.5%): 4.17%
